**Known issue:** this notebook attempts a Vision Transformer (`keras_cv.models.VisionTransformer`, `vit_b16_imagenet1k` preset). It never trained — `keras_cv` requires TensorFlow ≥ 2.11.0, and this project is pinned to TensorFlow 2.10.0. The notebook fails at model creation with `RuntimeError: The Tensorflow package version needs to be at least 2.11.0 for KerasCV to run.` See `prep/deepdive.md` for details.

In [2]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())


[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 1270783419449397439
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5738397696
locality {
  bus_id: 1
  links {
  }
}
incarnation: 98257525403631243
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 3070 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]


In [3]:
import tensorflow as tf
print(tf.__version__)


2.10.0


In [4]:
import tensorflow as tf
import keras
import keras_cv
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Define paths
data_train = "E:/Projects/Strabismus/Final_Train_data"
data_val = "E:/Projects/Strabismus/Split_Data/val"
data_test = "E:/Projects/Strabismus/Split_Data/test"

# Parameters
img_size = (224, 224)
batch_size = 32
epochs = 100
num_classes = 5

# Load datasets
train_dataset = image_dataset_from_directory(data_train, image_size=img_size, batch_size=batch_size)
val_dataset = image_dataset_from_directory(data_val, image_size=img_size, batch_size=batch_size)
test_dataset = image_dataset_from_directory(data_test, image_size=img_size, batch_size=batch_size, shuffle=False)

# Normalize data
autotune = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(autotune)
val_dataset = val_dataset.prefetch(autotune)
test_dataset = test_dataset.prefetch(autotune)

# Create ViT model
def create_vit_model(input_shape=(224, 224, 3), num_classes=5):
    vit = keras_cv.models.VisionTransformer.from_preset("vit_b16_imagenet1k", input_shape=input_shape, num_classes=num_classes)
    return vit

model = create_vit_model()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Train model
history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs)

# Evaluate model
y_true = np.concatenate([y for x, y in test_dataset], axis=0)
y_pred_probs = model.predict(test_dataset)
y_pred = np.argmax(y_pred_probs, axis=1)

# Compute metrics
print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(num_classes), yticklabels=range(num_classes))
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Plot learning curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Accuracy Curve")

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Loss Curve")
plt.show()


RuntimeError: The Tensorflow package version needs to be at least 2.11.0 for KerasCV to run. Currently, your TensorFlow version is 2.10.0. Please upgrade with `$ pip install --upgrade tensorflow`. You can use `pip freeze` to check afterwards that everything is ok.